## JSON Parsing and Processing

In [2]:
import json
import os
os.makedirs("data/json_files", exist_ok=True)

In [3]:
# Sample Nested JSON Data
data = {
    "products": [
        {"id": 1, "name": "Laptop", "price": 999.99},
        {"id": 2, "name": "Smartphone", "price": 499.99},
        {"id": 3, "name": "Tablet", "price": 299.99}
    ],
    "store": {
        "name": "Tech Store",
        "location": "123 Tech Street"
    }
}

In [ ]:
# Save JSON to file
json_file_path = "data/json_files/products.json"

with open(json_file_path, 'w') as json_file:
    json.dump(data, json_file, indent=2)


In [6]:
# Create and save jsonl file
jsonl_file_path = "data/json_files/products.jsonl"
with open(jsonl_file_path, 'w') as jsonl_file:
    for product in data["products"]:
        jsonl_file.write(json.dumps(product) + "\n")
        

### JSON Processing Techniques

In [9]:
from langchain_community.document_loaders import JSONLoader

# Methond 1: Load JSON file with jq_schema
print("Loading JSON with jq_schema...")

# Extract only the products array using jq_schema
data_loader = JSONLoader(
    file_path=json_file_path,
    jq_schema=".products[]",
    text_content = False
)

documents = data_loader.load()
print(f"Loaded {len(documents)} documents from JSON file with jq_schema.")
print("Sample document:", documents[0].page_content)
print(documents)


Loading JSON with jq_schema...
Loaded 3 documents from JSON file with jq_schema.
Sample document: {"id": 1, "name": "Laptop", "price": 999.99}
[Document(metadata={'source': '/Users/namitkumar/programming/rag/notebooks/data-parsing/data/json_files/products.json', 'seq_num': 1}, page_content='{"id": 1, "name": "Laptop", "price": 999.99}'), Document(metadata={'source': '/Users/namitkumar/programming/rag/notebooks/data-parsing/data/json_files/products.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Smartphone", "price": 499.99}'), Document(metadata={'source': '/Users/namitkumar/programming/rag/notebooks/data-parsing/data/json_files/products.json', 'seq_num': 3}, page_content='{"id": 3, "name": "Tablet", "price": 299.99}')]


In [18]:
# Custom JSON parsing
print("\nLoading JSON with custom parsing...")
from typing import List
from langchain_core.documents import Document

def custom_json_parser(file_path: str) -> List[Document]:
    with open(file_path, 'r') as json_file:
        data = json.load(json_file)
    
    documents = []
    for product in data["products"]:
        content = f"Product ID: {product['id']}\nName: {product['name']}\nPrice: ${product['price']}"
        metadata = {"source": file_path, "product_id": product['id']}
        documents.append(Document(page_content=content, metadata=metadata))
    
    return documents    

custom_documents = custom_json_parser(json_file_path)
print(f"Loaded {len(custom_documents)} documents from JSON file with custom parsing.")
print("Sample document:", custom_documents[0].page_content)
print(custom_documents)



Loading JSON with custom parsing...
Loaded 3 documents from JSON file with custom parsing.
Sample document: Product ID: 1
Name: Laptop
Price: $999.99
[Document(metadata={'source': 'data/json_files/products.json', 'product_id': 1}, page_content='Product ID: 1\nName: Laptop\nPrice: $999.99'), Document(metadata={'source': 'data/json_files/products.json', 'product_id': 2}, page_content='Product ID: 2\nName: Smartphone\nPrice: $499.99'), Document(metadata={'source': 'data/json_files/products.json', 'product_id': 3}, page_content='Product ID: 3\nName: Tablet\nPrice: $299.99')]
